# **Tutorial: Guía Interactiva de Qwen3-TTS**

**Autor:** Gemini CLI
**Modelo:** Qwen3-TTS (Versiones 1.7B y 0.6B)
**Tarea:** Texto a Voz (TTS), Clonación de Voz, Diseño de Voz

Este notebook demuestra las capacidades de **Qwen3-TTS**, un modelo de generación de voz de última generación. Exploraremos:
1.  **Voz Personalizada (Custom Voice):** Generación de voz utilizando personajes preestablecidos de alta calidad.
2.  **Diseño de Voz (Voice Design):** Creación de voces totalmente nuevas utilizando descripciones en lenguaje natural.
3.  **Clonación de Voz (Voice Cloning):** Clonación de una voz a partir de un clip de audio de referencia corto (Zero-Shot).

---

![image.png](attachment:image.png)

## **Paso 1: Instalación y Configuración**

Primero, necesitamos instalar el paquete `qwen-tts`. También instalaremos `flash-attn` para una generación más rápida, aunque requiere una GPU (se recomienda T4 o superior).

*Nota: La compilación de `flash-attn` puede tardar unos minutos. Si tienes prisa o estás en una CPU, puedes omitirlo, pero debes eliminar `attn_implementation="flash_attention_2"` del código de carga del modelo posterior.*

In [ ]:
# @title Instalar Dependencias

# ==============================================================================
# 🚀 INSTALACIÓN OPTIMIZADA PARA QWEN-TTS EN GOOGLE COLAB
# ==============================================================================
# Nota: La instalación es un proceso CPU-bound. La GPU se utilizará al ejecutar el modelo.
# ==============================================================================

import os, sys, subprocess, time

# 1. Configurar variables de entorno para compilación paralela acelerada
os.environ['MAX_JOBS'] = '4'  # Máximo de procesos de compilación simultáneos
os.environ['NINJA_JOBS'] = '4'  # Procesos paralelos para Ninja
os.environ['TORCH_CUDA_ARCH_LIST'] = '7.0 7.5 8.0 8.6 9.0'  # Arquitecturas CUDA compatibles

print("🔧 Configurando entorno de compilación optimizado...")

# 2. Instalar dependencias del sistema (solo si es necesario)
!apt-get update -qq > /dev/null 2>&1
!apt-get install -y -qq ninja-build > /dev/null 2>&1

# 3. Instalar dependencias principales de Python (modo silencioso y sin caché para evitar conflictos)
print("📦 Instalando dependencias base (qwen-tts, transformers, accelerate)...")
!pip install -q --no-cache-dir qwen-tts > /dev/null 2>&1

# 4. Intentar instalar flash-attn con tiempo de espera controlado (fallback seguro)
print("⚡ Intentando instalar flash-attn (optimización de atención)...")
print("   (Si tarda más de ~5 minutos, se omitirá automáticamente sin afectar la funcionalidad básica)")

try:
    # Ejecutar con límite de tiempo aproximado mediante timeout de shell
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
         "flash-attn", "--no-build-isolation"],
        timeout=300,  # 5 minutos máximo para esta compilación
        capture_output=True,
        text=True
    )
    if result.returncode == 0:
        print("✅ flash-attn instalado correctamente.")
    else:
        print("⚠️ flash-attn: compilación fallida o lenta. Se omitirá (el modelo funcionará con atención estándar).")
except subprocess.TimeoutExpired:
    print("⏱️ flash-attn: tiempo de compilación excedido. Omitiendo para continuar con la instalación.")
except Exception as e:
    print(f"⚠️ flash-attn: error inesperado ({e}). Continuando sin este módulo opcional.")

# 5. Verificación final del entorno
print("\n🔍 Verificando instalación...")
try:
    import torch
    print(f"✅ PyTorch: {torch.__version__} | GPU disponible: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"   └─ GPU: {torch.cuda.get_device_name(0)}")

    import qwen_tts
    print(f"✅ qwen-tts: {qwen_tts.__version__}")

    import accelerate
    print(f"✅ accelerate: {accelerate.__version__}")

    # Verificar flash-attn opcionalmente
    try:
        import flash_attn
        print(f"✅ flash-attn: {flash_attn.__version__} (optimización activa)")
    except ImportError:
        print("⚠️ flash-attn: no instalado (se usará atención estándar de PyTorch)")

    print("\n🎉 ¡Entorno listo para generar audio con Qwen-TTS!")

except ImportError as e:
    print(f"❌ Error crítico en la instalación: {e}")
    print("💡 Sugerencia: Reinie el entorno de ejecución (Menú: Entorno de ejecución > Reiniciar) e intente nuevamente.")

🔧 Configurando entorno de compilación optimizado...
📦 Instalando dependencias base (qwen-tts, transformers, accelerate)...
⚡ Intentando instalar flash-attn (optimización de atención)...
   (Si tarda más de ~5 minutos, se omitirá automáticamente sin afectar la funcionalidad básica)
⏱️ flash-attn: tiempo de compilación excedido. Omitiendo para continuar con la instalación.

🔍 Verificando instalación...
✅ PyTorch: 2.10.0+cu128 | GPU disponible: True
   └─ GPU: Tesla T4


AttributeError: module 'qwen_tts' has no attribute '__version__'

In [3]:
# @title Importar Librerías y Verificar GPU
import torch
import soundfile as sf
from IPython.display import Audio, display
import os

# Verificar si CUDA (GPU) está disponible
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando dispositivo: {device}")

if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Advertencia: Ejecutando en CPU. La generación será lenta.")

# Función auxiliar para reproducir audio directamente en el notebook
def play_audio(file_path):
    if os.path.exists(file_path):
        display(Audio(file_path))
    else:
        print(f"Archivo no encontrado: {file_path}")

Usando dispositivo: cuda
GPU: Tesla T4


---

## **Paso 2: Generación de Voz Personalizada (Personajes Preestablecidos)**

Qwen3-TTS viene con varias voces preestablecidas de alta calidad (por ejemplo, Vivian, Ryan, Tanaka). Este modo es mejor para texto a voz general donde deseas un personaje específico y estable.

### **Cargar Modelo**
Usaremos el modelo `Qwen3-TTS-12Hz-1.7B-CustomVoice`.

In [5]:
# @title Cargar Modelo Qwen-TTS (Compatible sin flash-attn)
import torch
import gc

# 1. Liberar memoria residual antes de cargar el modelo
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# 2. Definir dispositivo y precisión
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if device == "cuda" else torch.float32
print(f"🚀 Dispositivo: {device} | Precisión: {dtype}")

# 3. Determinar implementación de atención disponible
try:
    import flash_attn
    attn_impl = "flash_attention_2"
    print("✅ flash-attn detectado: usando optimización avanzada")
except ImportError:
    attn_impl = "sdpa"  # Scaled Dot-Product Attention (nativo de PyTorch)
    print("ℹ️ flash-attn no disponible: usando atención estándar (sdpa)")

# 4. Seleccionar modelo
model_name = "Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice"  # @param ["Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice", "Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice"]
print(f"📥 Cargando: {model_name}")

# 5. Cargar modelo con configuración compatible
from qwen_tts import Qwen3TTSModel

custom_voice_model = Qwen3TTSModel.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=dtype,
    attn_implementation=attn_impl,  # Valor dinámico según disponibilidad
    low_cpu_mem_usage=True,         # Optimización de RAM durante carga
)

print("✅ ¡Modelo de Voz Personalizada cargado exitosamente!")

`torch_dtype` is deprecated! Use `dtype` instead!


🚀 Dispositivo: cuda | Precisión: torch.bfloat16
ℹ️ flash-attn no disponible: usando atención estándar (sdpa)
📥 Cargando: Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice


model.safetensors:   0%|          | 0.00/3.83G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/245 [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

preprocessor_config.json:   0%|          | 0.00/127 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

✅ ¡Modelo de Voz Personalizada cargado exitosamente!


### **Generar Voz**
Hagamos que "Vivian" (Chino) o "Ryan" (Inglés) hablen. También puedes proporcionar una instrucción (`instruct`) para modificar la emoción.

![image.png](attachment:image.png)

In [6]:
# @title Generar Voces Preestablecidas (Optimizado)
import torch
import gc
import soundfile as sf  # Requerido para sf.write
from IPython.display import Audio, display  # Para reproducción en Colab

# 1. Liberar memoria antes de la generación
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# 2. Parámetros de generación
text_input = "¡Hola! Esta es una demostración de Qwen3-TTS. Puedo hablar con emociones claras."
speaker_name = "Dylan"  # @param ["Vivian", "Serena", "Uncle_Fu", "Dylan", "Eric", "Ryan", "Aiden", "Ono_Anna", "Sohee"]
instruction = "Feliz y enérgica"
language = "Spanish"  # Detección automática disponible, pero se recomienda especificar

print(f"🎙️ Generando audio para {speaker_name} con instrucción: '{instruction}'")

# 3. Generación de audio con manejo de contexto para eficiencia de memoria
with torch.inference_mode():  # Desactiva gradientes para ahorrar memoria
    wavs, sr = custom_voice_model.generate_custom_voice(
        text=text_input,
        language=language,
        speaker=speaker_name,
        instruct=instruction,
        # Parámetros opcionales para mayor control:
        # temperature=0.7,      # Controla creatividad/variabilidad (0.0-1.0)
        # top_p=0.95,           # Muestreo nuclear para calidad consistente
        # max_new_tokens=2048,  # Límite de longitud de audio generado
    )

# 4. Guardar archivo de audio
output_path = f"output_{speaker_name}.wav"
sf.write(output_path, wavs[0], sr)
print(f"✅ Audio guardado en: {output_path}")

# 5. Reproducir directamente en Colab
print("🔊 Reproduciendo audio...")
display(Audio(output_path, autoplay=True))

# 6. Liberar memoria post-generación
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("🧹 Memoria liberada. Listo para la siguiente generación.")

🎙️ Generando audio para Dylan con instrucción: 'Feliz y enérgica'


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


✅ Audio guardado en: output_Dylan.wav
🔊 Reproduciendo audio...


🧹 Memoria liberada. Listo para la siguiente generación.


---

## **Paso 3: Diseño de Voz (Texto a Voz)**

Esta función te permite **describir** una voz utilizando lenguaje natural, y el modelo generará un habla que coincida con esa descripción. Esto es poderoso para crear NPCs o personajes específicos sin audio de referencia.

### **Cargar Modelo**
Necesitamos cargar una variante específica del modelo para esto: `Qwen3-TTS-12Hz-1.7B-VoiceDesign`.

In [5]:
# @title Cargar Modelo Voice Design
import gc
import torch

# Limpiar modelo anterior para ahorrar VRAM
del custom_voice_model
gc.collect()
torch.cuda.empty_cache()

# Cargar el modelo Voice Design
voice_design_model = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign",
    device_map="auto",
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    attn_implementation="sdpa",  # Atención nativa de PyTorch (segura sin flash-attn)
)

print("¡Modelo de Diseño de Voz Cargado!")

NameError: name 'custom_voice_model' is not defined

### **Diseñar una Voz**
Intenta describir edad, género, tono, velocidad de habla y emoción.

In [8]:
# @title Generar Voz Personalizada (Voice Design)
import torch
import soundfile as sf
from IPython.display import Audio, display

text_to_speak = "No puedo creer que finalmente llegamos a la cima de la montaña. ¡La vista es increíble!"
# Descripción: Un hombre de mediana edad con voz grave y profunda, hablando lentamente y sonando exhausto pero asombrado.
voice_description = '''gender: Male
pitch: Deep and resonant with subtle downward inflections suggesting gravity
speed: Deliberately slow with extended pauses between sentences
volume: Moderate to soft, creating an intimate atmosphere
age: Middle-aged to older adult
clarity: Crystal clear enunciation with careful articulation
fluency: Smooth and controlled with intentional dramatic pauses
accent: Standard American English
texture: Rich and velvety with a slightly smoky quality
emotion: Contemplative and intriguing
tone: Mysterious, philosophical, and atmospheric
personality: Introspective, wise, and captivating '''

print("Diseñando voz y generando...")

# Ejecución en modo inferencia para mayor eficiencia y menor consumo de VRAM
with torch.inference_mode():
    wavs, sr = voice_design_model.generate_voice_design(
        text=text_to_speak,
        language="Spanish",
        instruct=voice_description
    )

output_design_path = "output_voice_design.wav"
sf.write(output_design_path, wavs[0], sr)

print(f"✅ Audio guardado en: {output_design_path}")
display(Audio(output_design_path, autoplay=True))

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


Diseñando voz y generando...
✅ Audio guardado en: output_voice_design.wav


In [4]:
# @title Generar Voz Personalizada (Perfil Femenino/Meditativo)
import torch
import soundfile as sf
from IPython.display import Audio, display

# Texto a sintetizar
text_to_speak = "No puedo creer que finalmente llegamos a la cima de la montaña. ¡La vista es increíble!"

# Descripción detallada del perfil de voz (en inglés para mayor precisión del modelo)
voice_description = '''gender: Female
pitch: Medium-low female pitch with gentle, soothing fluctuations
speed: Very slow and measured, allowing time for mental processing
volume: Soft and calming, never raising above comfortable levels
age: Adult (30s-40s)
clarity: Exceptionally clear with soft consonants
fluency: Perfectly fluid with mindful breathing pauses
accent: Neutral North American with slight California influence
texture: Warm and breathy, incredibly smooth
emotion: Peaceful and nurturing
tone: Gentle, encouraging, and meditative
personality: Compassionate, patient, and serene'''

print("🎨 Diseñando voz y generando audio...")

# Ejecución en modo inferencia para eficiencia de memoria
with torch.inference_mode():
    wavs, sr = voice_design_model.generate_voice_design(
        text=text_to_speak,
        language="Spanish",  # Idioma del texto de salida
        instruct=voice_description  # Instrucciones de diseño de voz
    )

# Guardar archivo
output_design_path = "output_voice_design_female.wav"
sf.write(output_design_path, wavs[0], sr)

print(f"✅ Audio guardado en: {output_design_path}")
display(Audio(output_design_path, autoplay=True))

🎨 Diseñando voz y generando audio...


NameError: name 'voice_design_model' is not defined

In [3]:
# @title Generar Voz Personalizada (Perfil Infantil 8-10 años)
import torch
import gc
import soundfile as sf
from IPython.display import Audio, display

# 1. Verificar si el modelo está cargado; si no, cargarlo automáticamente
try:
    # Intenta usar la variable existente
    model_to_use = voice_design_model
    print("✅ Utilizando modelo 'voice_design_model' ya cargado.")
except NameError:
    print("⚠️ Modelo no encontrado en memoria. Iniciando carga automática...")
    from qwen_tts import Qwen3TTSModel

    # Detectar dispositivo disponible
    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.bfloat16 if device == "cuda" else torch.float32

    print(f"🚀 Cargando modelo para: {device.upper()} | Precisión: {dtype}")

    # Cargar modelo con configuración segura para CPU/GPU
    model_to_use = Qwen3TTSModel.from_pretrained(
        "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign",
        device_map="auto",
        torch_dtype=dtype,
        attn_implementation="sdpa",  # Compatible nativamente con CPU y GPU
        low_cpu_mem_usage=True
    )
    print("✅ Modelo cargado exitosamente.")

# 2. Parámetros de Generación
text_to_speak = "No puedo creer que finalmente llegamos a la cima de la montaña. ¡La vista es increíble!"

voice_description = '''gender: Male
pitch: High child's voice with wide pitch variations for storytelling
speed: Variable - rushing through exciting parts, slowing for details
volume: Moderate with sudden louder bursts during exciting moments
age: Child (8-10 years old)
clarity: Generally clear but with occasional word stumbles
fluency: Enthusiastic flow with natural childlike interruptions
accent: American English (General American)
texture: Bright and youthful with slight breathiness
emotion: Wonder and excitement mixed with nervousness
tone: Animated, imaginative, and earnest
personality: Innocent, creative, and eager to share'''

print("🎨 Diseñando voz infantil y generando audio...")

# 3. Generación de Audio
# torch.inference_mode() optimiza la memoria tanto en CPU como en GPU
with torch.inference_mode():
    wavs, sr = model_to_use.generate_voice_design(
        text=text_to_speak,
        language="Spanish",
        instruct=voice_description
    )

# 4. Guardar y Reproducir
output_design_path = "output_voice_design_child.wav"
sf.write(output_design_path, wavs[0], sr)

print(f"✅ Audio guardado en: {output_design_path}")
display(Audio(output_design_path, autoplay=True))

⚠️ Modelo no encontrado en memoria. Iniciando carga automática...


ModuleNotFoundError: No module named 'qwen_tts'

---

## **Paso 4: Clonación de Voz (Zero-Shot)**

El modelo "Base" te permite clonar cualquier voz utilizando un clip de audio de referencia corto (3-10 segundos).

### **Cargar Modelo**
Usamos `Qwen3-TTS-12Hz-1.7B-Base`.

In [ ]:
# Limpiar memoria
del voice_design_model
torch.cuda.empty_cache()

# @title Seleccionar y Cargar Modelo Base
model_name = "Qwen/Qwen3-TTS-12Hz-1.7B-Base" # @param ["Qwen/Qwen3-TTS-12Hz-1.7B-Base", "Qwen/Qwen3-TTS-12Hz-0.6B-Base"]

print(f"Cargando {model_name}...")

# Cargar el modelo Base/Clone
base_model = Qwen3TTSModel.from_pretrained(
    model_name,
    device_map="auto",
    dtype=torch.bfloat16 if device == "cuda" else torch.float32,
    attn_implementation="flash_attention_2" if device == "cuda" else None,
)

print("¡Modelo de Clonación de Voz Cargado!")

Cargando Qwen/Qwen3-TTS-12Hz-1.7B-Base...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/245 [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

preprocessor_config.json:   0%|          | 0.00/127 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

¡Modelo de Clonación de Voz Cargado!


### **Clonar una Voz**
Puedes usar una URL a un archivo de audio o subir uno a Colab. Aquí usamos una URL de ejemplo del repositorio.

In [ ]:
# Puedes reemplazar esta URL con una ruta a un archivo local (ej. "/content/mi_voz.wav")
ref_audio_path = "https://drive.google.com/uc?export=download&id=1dYl1634xT4UBAclsNboRMCbq-7EWTOj_"
ref_audio_text = "Hola, esta es una prueba de mi voz... el objetivo es ver si este modelo puede realmente clonar mi voz como espero que lo haga, y  suene ¡Al menos! muy similar a mi"

print(f"Clonando voz desde referencia...")

# 1. Crear estructura del prompt (opcional pero recomendado para reusar)
# Esto extrae las características de estilo del audio de referencia
voice_clone_prompt = base_model.create_voice_clone_prompt(
    ref_audio=ref_audio_path,
    ref_text=ref_audio_text
)


Clonando voz desde referencia...


### **Haciendo inferencia de la voz clonada**

In [ ]:
# 2. Generar - Spanish example
target_text = "Esto es lo que sucede cuando clonas una voz. El parecido es bastante asombroso, ¿no?"

wavs, sr = base_model.generate_voice_clone(
    text=target_text,
    language="Spanish",
    voice_clone_prompt=voice_clone_prompt
)

output_clone_path = "output_cloned.wav"
sf.write(output_clone_path, wavs[0], sr)

print("Referencia Original:")
# Si es una URL, no podemos reproducirla directamente con IPython.display(Audio(url)) fácilmente en todos los navegadores sin descargar,
# pero para archivos locales funciona.
print(f"(Fuente de referencia: {ref_audio_path})")

print("\nResultado Clonado:")
play_audio(output_clone_path)

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


Referencia Original:
(Fuente de referencia: https://drive.google.com/uc?export=download&id=1dYl1634xT4UBAclsNboRMCbq-7EWTOj_)

Resultado Clonado:


In [ ]:
# 2. Generar - English example
target_text = "This is another example of voice cloning. The resemblance is quite amazing, isn't it?"

wavs, sr = base_model.generate_voice_clone(
    text=target_text,
    language="English",
    voice_clone_prompt=voice_clone_prompt
)

output_clone_path = "output_cloned.wav"
sf.write(output_clone_path, wavs[0], sr)

print("Referencia Original:")
# Si es una URL, no podemos reproducirla directamente con IPython.display(Audio(url)) fácilmente en todos los navegadores sin descargar,
# pero para archivos locales funciona.
print(f"(Fuente de referencia: {ref_audio_path})")

print("\nResultado Clonado:")
play_audio(output_clone_path)

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


Referencia Original:
(Fuente de referencia: https://drive.google.com/uc?export=download&id=1dYl1634xT4UBAclsNboRMCbq-7EWTOj_)

Resultado Clonado:


---

## **Paso 5: Modelos Ligeros (0.6B)**

Los modelos de 0.6B son significativamente más rápidos y requieren menos VRAM. Son ideales para prototipado rápido o hardware limitado.

Primero, **limpiaremos la memoria de la GPU** para asegurarnos de tener espacio suficiente.

In [ ]:
# Limpiar memoria de la GPU
try:
    del base_model
except NameError:
    pass
try:
    del custom_voice_model
except NameError:
    pass
try:
    del voice_design_model
except NameError:
    pass

import gc
gc.collect()
torch.cuda.empty_cache()
print("Memoria GPU liberada.")

Memoria GPU liberada.


In [ ]:
# Cargar modelo 0.6B CustomVoice
from qwen_tts import Qwen3TTSModel

print("Cargando Qwen3-TTS-12Hz-0.6B-CustomVoice...")
model_06b = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice",
    device_map="auto",
    dtype=torch.bfloat16 if device == "cuda" else torch.float32,
    attn_implementation="flash_attention_2" if device == "cuda" else None,
)

Cargando Qwen3-TTS-12Hz-0.6B-CustomVoice...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.81G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/245 [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


Modelo 0.6B Cargado. Probando generación...


In [ ]:
print("Modelo 0.6B Cargado. Probando generación...")
text_to_speak = "No puedo creer que finalmente llegamos a la cima de la montaña. ¡La vista es increíble!"

wavs, sr = model_06b.generate_custom_voice(
    text=text_to_speak,
    language="Spanish",
    speaker="Ryan"
)

sf.write("output_06b.wav", wavs[0], sr)
play_audio("output_06b.wav")

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


Modelo 0.6B Cargado. Probando generación...


---

## **Resumen**

En este notebook, hemos cubierto:
1.  **CustomVoice:** Usando `generate_custom_voice` con presets como "Ryan" o "Vivian".
2.  **VoiceDesign:** Usando `generate_voice_design` para crear voces a partir de instrucciones de texto (ej. "Deep, scary monster voice").
3.  **VoiceClone:** Usando `generate_voice_clone` con el modelo Base para copiar una voz desde un archivo de referencia.

**Consejos para Mejores Resultados:**
*   **Ingeniería de Prompts:** Para el Diseño de Voz, sé descriptivo. Menciona género, edad, tono, velocidad y emoción (preferiblemente en inglés para el prompt de descripción).
*   **Audio de Referencia:** Para Clonación, usa audio limpio sin música de fondo. 3-10 segundos suelen ser suficientes.
*   **Idioma:** El modelo soporta 10 idiomas (Inglés, Chino, Japonés, Coreano, Alemán, Francés, Ruso, Portugués, Español, Italiano). Puedes cambiar el parámetro `language` según corresponda.